# 01. Data Check

merged_data.csv의 구조, 타입, 결측치 및 중복 여부를 확인합니다.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정 (Windows 기준)
plt.rc('font', family='Malgun Gothic') 
plt.rc('axes', unicode_minus=False)

# 데이터 불러오기
df = pd.read_csv('../merged_data.csv')

In [6]:
# 39개 컬럼명 모두 확인
print(df.columns.tolist())

['Time (h)', 'Aeration rate(Fg:L/h)', 'Agitator RPM(RPM:RPM)', 'Sugar feed rate(Fs:L/h)', 'Acid flow rate(Fa:L/h)', 'Base flow rate(Fb:L/h)', 'Heating/cooling water flow rate(Fc:L/h)', 'Heating water flow rate(Fh:L/h)', 'Water for injection/dilution(Fw:L/h)', 'Air head pressure(pressure:bar)', 'Dumped broth flow(Fremoved:L/h)', 'Substrate concentration(S:g/L)', 'Dissolved oxygen concentration(DO2:mg/L)', 'Penicillin concentration(P:g/L)', 'Vessel Volume(V:L)', 'Vessel Weight(Wt:Kg)', 'pH(pH:pH)', 'Temperature(T:K)', 'Generated heat(Q:kJ)', 'carbon dioxide percent in off-gas(CO2outgas:%)', 'PAA flow(Fpaa:PAA flow (L/h))', 'PAA concentration offline(PAA_offline:PAA (g L^{-1}))', 'Oil flow(Foil:L/hr)', 'NH_3 concentration off-line(NH3_offline:NH3 (g L^{-1}))', 'Oxygen Uptake Rate(OUR:(g min^{-1}))', 'Oxygen in percent in off-gas(O2:O2  (%))', 'Offline Penicillin concentration(P_offline:P(g L^{-1}))', 'Offline Biomass concentratio(X_offline:X(g L^{-1}))', 'Carbon evolution rate(CER:g/h)', 

In [10]:
# 제어전략(Strategy) 파생변수 생성 - Batch_ID 기준 매핑
def get_strategy(bid):
    if bid <= 30: return 'RC'
    elif bid <= 60: return 'OC'
    elif bid <= 90: return 'APC'
    else: return 'Fault'
df['Strategy'] = df['Batch_ID'].apply(get_strategy)

# 컬럼 그룹 정의 (X값 후보만)
core_cols = ['Penicillin concentration(P:g/L)', 'Substrate concentration(S:g/L)',
             'Dissolved oxygen concentration(DO2:mg/L)', 'Vessel Volume(V:L)',
             'pH(pH:pH)', 'Temperature(T:K)']

operating_cols = ['Agitator RPM(RPM:RPM)', 'Sugar feed rate(Fs:L/h)', 'Aeration rate(Fg:L/h)',
                  'Acid flow rate(Fa:L/h)', 'Base flow rate(Fb:L/h)',
                  'Heating/cooling water flow rate(Fc:L/h)', 'Heating water flow rate(Fh:L/h)',
                  'Water for injection/dilution(Fw:L/h)', 'PAA flow(Fpaa:PAA flow (L/h))',
                  'Oil flow(Foil:L/hr)', 'Air head pressure(pressure:bar)',
                  'Generated heat(Q:kJ)', 'Vessel Weight(Wt:Kg)', 'Dumped broth flow(Fremoved:L/h)',
                  'Ammonia shots(NH3_shots:kgs)', 'carbon dioxide percent in off-gas(CO2outgas:%)',
                  'Oxygen in percent in off-gas(O2:O2  (%))', 'Oxygen Uptake Rate(OUR:(g min^{-1}))',
                  'Carbon evolution rate(CER:g/h)']

offline_cols = ['Offline Biomass concentratio(X_offline:X(g L^{-1}))',
                'Offline Penicillin concentration(P_offline:P(g L^{-1}))',
                'PAA concentration offline(PAA_offline:PAA (g L^{-1}))',
                'NH_3 concentration off-line(NH3_offline:NH3 (g L^{-1}))',
                'Viscosity(Viscosity_offline:centPoise)']

y_cols = ['Penicllin_yield_total (kg)', 'Penicllin_harvested_end_of_batch (kg)',
          'Penicllin_harvested_during_batch(kg)']

▲ Batch_ID 범위(1-30/31-60/61-90/91-100) 기준으로 생성합니다. 이후 모든 분석에서 이 컬럼 그룹 변수를 재사용합니다.

# 1. 전체 구조 파악

In [11]:
print("=== 1. 데이터 기본 구조 ===")
print(f"전체 Shape: {df.shape}")
print(f"배치 수: {df['Batch_ID'].nunique()}개")
print()
print("[Strategy 그룹별 배치 수]")
print(df.groupby('Strategy')['Batch_ID'].nunique())
print()
print("[배치별 소요시간(Time) 분포]")
print(df.groupby('Batch_ID')['Time (h)'].max().describe())

=== 1. 데이터 기본 구조 ===
전체 Shape: (113935, 40)
배치 수: 100개

[Strategy 그룹별 배치 수]
Strategy
APC      30
Fault    10
OC       30
RC       30
Name: Batch_ID, dtype: int64

[배치별 소요시간(Time) 분포]
count    100.000000
mean     227.870000
std       18.150691
min      167.000000
25%      227.500000
50%      230.000000
75%      230.000000
max      290.000000
Name: Time (h), dtype: float64


# 2. 결측치 확인

In [12]:
print("=== 2. 결측치 확인 ===")
missing_info = df.isnull().sum()
missing_info = missing_info[missing_info > 0].sort_values(ascending=False)
print("[결측치 발생 컬럼]")
print(missing_info)
print()
print("[오프라인 변수 결측 비율(%)]")
print((df[offline_cols].isnull().mean() * 100).round(1))

=== 2. 결측치 확인 ===
[결측치 발생 컬럼]
PAA concentration offline(PAA_offline:PAA (g L^{-1}))      111873
NH_3 concentration off-line(NH3_offline:NH3 (g L^{-1}))    111873
Offline Penicillin concentration(P_offline:P(g L^{-1}))    111873
Offline Biomass concentratio(X_offline:X(g L^{-1}))        111873
Viscosity(Viscosity_offline:centPoise)                     111873
dtype: int64

[오프라인 변수 결측 비율(%)]
Offline Biomass concentratio(X_offline:X(g L^{-1}))        98.2
Offline Penicillin concentration(P_offline:P(g L^{-1}))    98.2
PAA concentration offline(PAA_offline:PAA (g L^{-1}))      98.2
NH_3 concentration off-line(NH3_offline:NH3 (g L^{-1}))    98.2
Viscosity(Viscosity_offline:centPoise)                     98.2
dtype: float64
